# Training Together (Minimal Loop)

This notebook runs a minimal end-to-end training loop using your existing implementation.
It supports configurable hyperparameters, `np.memmap` dataset loading, periodic train/val logging, and checkpoint serialization.

Default config is set to TinyStories raw `.txt` files in `data/` using byte-level training.

Local logging: TensorBoard is enabled by default.


In [ ]:
%load_ext autoreload
%autoreload 2

import argparse
import time
from pathlib import Path

import numpy as np
import torch
from src.transformer.transformer_lm import TransformerLM
from src.training.optimizer import AdamW
from src.training.loss import cross_entropy
from src.training_loop.data_loading import get_batch
from src.training_loop.checkpointing import save_checkpoint, load_checkpoint
from torch.utils.tensorboard import SummaryWriter


In [ ]:
def parse_config(argv=None):
    parser = argparse.ArgumentParser(add_help=False)

    # Data
    parser.add_argument("--train_data", type=str, default="../data/TinyStoriesV2-GPT4-train.txt")
    parser.add_argument("--val_data", type=str, default="../data/TinyStoriesV2-GPT4-valid.txt")
    parser.add_argument("--memmap_dtype", type=str, default="uint8")

    # Model
    parser.add_argument("--vocab_size", type=int, default=256)
    parser.add_argument("--context_length", type=int, default=128)
    parser.add_argument("--d_model", type=int, default=320)
    parser.add_argument("--num_heads", type=int, default=8)
    parser.add_argument("--d_ff", type=int, default=1280)
    parser.add_argument("--num_layers", type=int, default=6)
    parser.add_argument("--theta", type=float, default=10000.0)

    # Optimization
    parser.add_argument("--batch_size", type=int, default=24)
    parser.add_argument("--learning_rate", type=float, default=3e-4)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--beta1", type=float, default=0.9)
    parser.add_argument("--beta2", type=float, default=0.95)
    parser.add_argument("--eps", type=float, default=1e-8)
    parser.add_argument("--max_iters", type=int, default=3500)

    # Logging / eval / checkpoint
    parser.add_argument("--eval_interval", type=int, default=250)
    parser.add_argument("--eval_batches", type=int, default=10)
    parser.add_argument("--log_interval", type=int, default=25)
    parser.add_argument("--checkpoint_interval", type=int, default=500)
    parser.add_argument("--checkpoint_path", type=str, default="checkpoints/minimal_ckpt.pt")
    parser.add_argument("--resume", action="store_true")

    # Runtime
    parser.add_argument("--device", type=str, default=("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")))
    parser.add_argument("--seed", type=int, default=1337)

    # Optional external logging

    parser.add_argument("--use_tensorboard", action="store_true")
    parser.add_argument("--tb_logdir", type=str, default="runs/training_together")

    # In notebook mode, parse known args so Jupyter flags don't crash parsing.
    args, _ = parser.parse_known_args(argv)
    return args

# Use notebook-safe defaults (tuned for ~1h on M2). You can override by editing this list.
# Example:
# cfg = parse_config(["--max_iters", "1000", "--batch_size", "32"]) 
cfg = parse_config(["--use_tensorboard", "--device", "mps"])
print(cfg)


In [ ]:
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

if not Path(cfg.train_data).exists() or not Path(cfg.val_data).exists():
    raise FileNotFoundError(
        "Data files not found. Set --train_data and --val_data to your TinyStories paths (or tokenized memmaps)."
    )

dtype = np.dtype(cfg.memmap_dtype)
train_tokens = np.memmap(cfg.train_data, mode="r", dtype=dtype)
val_tokens = np.memmap(cfg.val_data, mode="r", dtype=dtype)

if len(train_tokens) <= cfg.context_length + 1 or len(val_tokens) <= cfg.context_length + 1:
    raise ValueError("Dataset is too small for the selected --context_length.")

print(f"Loaded train memmap: {cfg.train_data} ({len(train_tokens):,} tokens)")
print(f"Loaded val memmap:   {cfg.val_data} ({len(val_tokens):,} tokens)")


In [ ]:
device = torch.device(cfg.device)

model = TransformerLM(
    d_model=cfg.d_model,
    num_heads=cfg.num_heads,
    d_ff=cfg.d_ff,
    vocab_size=cfg.vocab_size,
    context_length=cfg.context_length,
    num_layers=cfg.num_layers,
    theta=cfg.theta,
    device=device,
)
model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    betas=(cfg.beta1, cfg.beta2),
    eps=cfg.eps,
)

start_iter = 0
if cfg.resume and Path(cfg.checkpoint_path).exists():
    start_iter = load_checkpoint(cfg.checkpoint_path, model, optimizer)
    print(f"Resumed from {cfg.checkpoint_path} at iteration {start_iter}")


In [ ]:
@torch.no_grad()
def estimate_split_loss(split_tokens, eval_batches):
    model.eval()
    losses = []
    for _ in range(eval_batches):
        x, y = get_batch(split_tokens, cfg.batch_size, cfg.context_length, cfg.device)
        out = model(x)  # (B, T, V)
        B, T, V = out.shape
        loss = cross_entropy(out.view(B * T, V), y.view(B * T))
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


In [ ]:
tb_writer = None
if cfg.use_tensorboard:
    Path(cfg.tb_logdir).mkdir(parents=True, exist_ok=True)
    tb_writer = SummaryWriter(log_dir=cfg.tb_logdir)
    print(f"TensorBoard logging to: {cfg.tb_logdir}")


In [ ]:
Path(cfg.checkpoint_path).parent.mkdir(parents=True, exist_ok=True)

model.train()
last_time = time.time()

for it in range(start_iter, cfg.max_iters):
    x, y = get_batch(train_tokens, cfg.batch_size, cfg.context_length, cfg.device)

    probs = model(x)  # (B, T, V)
    B, T, V = probs.shape
    loss = cross_entropy(probs.view(B * T, V), y.view(B * T))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if (it + 1) % cfg.log_interval == 0:
        now = time.time()
        dt = now - last_time
        last_time = now
        print(f"iter {it+1:6d} | train_loss {loss.item():.4f} | dt {dt:.2f}s")
        if tb_writer is not None:
            tb_writer.add_scalar("train/loss", loss.item(), it + 1)

    if (it + 1) % cfg.eval_interval == 0:
        train_eval = estimate_split_loss(train_tokens, cfg.eval_batches)
        val_eval = estimate_split_loss(val_tokens, cfg.eval_batches)
        print(f"iter {it+1:6d} | train_eval {train_eval:.4f} | val_eval {val_eval:.4f}")
        if tb_writer is not None:
            tb_writer.add_scalar("eval/train_loss", train_eval, it + 1)
            tb_writer.add_scalar("eval/val_loss", val_eval, it + 1)

    if (it + 1) % cfg.checkpoint_interval == 0 or (it + 1) == cfg.max_iters:
        save_checkpoint(model, optimizer, it + 1, cfg.checkpoint_path)
        print(f"saved checkpoint: {cfg.checkpoint_path} @ iter {it+1}")

if tb_writer is not None:
    tb_writer.close()
